# ExecutionBacktester Demo
This notebook demonstrates the event-driven backtesting framework.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
from backtester.core.engine import EventEngine
from backtester.core.data_loader import DataLoader
from backtester.core.portfolio import Portfolio
from backtester.core.execution import ExecutionSimulator
from backtester.core.performance import PerformanceAnalyzer
from backtester.core.events import MarketEvent, SignalEvent, OrderEvent, FillEvent
from backtester.strategies import MovingAverageStrategy, BreakoutStrategy

## 1. Initialize Components

In [ ]:
# Create event engine
engine = EventEngine()

# Load data
data_loader = DataLoader('../data/sample_data.csv', 'csv')
data_loader.load()

print(f"Loaded {len(data_loader.data)} market events")

## 2. Setup Portfolio and Execution

In [ ]:
# Initialize portfolio
portfolio = Portfolio(engine, initial_capital=100000)

# Initialize execution simulator
execution = ExecutionSimulator(engine, slippage_bps=5.0, commission_bps=1.0)

## 3. Create Strategy

In [ ]:
# Create moving average strategy
strategy = MovingAverageStrategy('ma_strategy', engine, short_window=20, long_window=50)

# Register event handlers
engine.register_handler(MarketEvent, strategy.on_market_event)
engine.register_handler(SignalEvent, portfolio.on_signal_event)
engine.register_handler(OrderEvent, execution.on_order_event)
engine.register_handler(FillEvent, portfolio.on_fill_event)

## 4. Run Backtest

In [ ]:
# Run backtest loop
while data_loader.has_more_data():
    event = data_loader.get_next_event()
    if event:
        engine.put(event)
        engine.process_events()
        
        # Update prices
        prices = data_loader.get_latest_prices()
        portfolio.update_holdings(prices)
        execution.update_prices(prices)

print("Backtest completed!")

## 5. Analyze Performance

In [ ]:
# Create performance analyzer
analyzer = PerformanceAnalyzer(portfolio.history, portfolio.initial_capital)
summary = analyzer.get_summary()

print("\n=== Performance Summary ===")
for key, value in summary.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

## 6. Visualize Results

In [ ]:
# Plot portfolio value over time
df = pd.DataFrame(portfolio.history)

plt.figure(figsize=(12, 6))
plt.plot(df['timestamp'], df['portfolio_value'])
plt.title('Portfolio Value Over Time')
plt.xlabel('Date')
plt.ylabel('Portfolio Value ($)')
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Plot returns distribution
returns = analyzer.calculate_returns()

plt.figure(figsize=(12, 6))
plt.hist(returns, bins=50, edgecolor='black')
plt.title('Returns Distribution')
plt.xlabel('Return')
plt.ylabel('Frequency')
plt.grid(True)
plt.tight_layout()
plt.show()

## 7. Compare Strategies

In [ ]:
# Run breakout strategy for comparison
engine2 = EventEngine()
data_loader2 = DataLoader('../data/sample_data.csv', 'csv')
data_loader2.load()
portfolio2 = Portfolio(engine2, initial_capital=100000)
execution2 = ExecutionSimulator(engine2)
strategy2 = BreakoutStrategy('breakout', engine2, lookback=20)

engine2.register_handler(MarketEvent, strategy2.on_market_event)
engine2.register_handler(SignalEvent, portfolio2.on_signal_event)
engine2.register_handler(OrderEvent, execution2.on_order_event)
engine2.register_handler(FillEvent, portfolio2.on_fill_event)

while data_loader2.has_more_data():
    event = data_loader2.get_next_event()
    if event:
        engine2.put(event)
        engine2.process_events()
        prices = data_loader2.get_latest_prices()
        portfolio2.update_holdings(prices)
        execution2.update_prices(prices)

analyzer2 = PerformanceAnalyzer(portfolio2.history, portfolio2.initial_capital)
summary2 = analyzer2.get_summary()

print("\n=== Strategy Comparison ===")
print(f"Moving Average - Return: {summary['total_return']:.2%}, Sharpe: {summary['sharpe_ratio']:.2f}")
print(f"Breakout - Return: {summary2['total_return']:.2%}, Sharpe: {summary2['sharpe_ratio']:.2f}")